# Setup

Use this notebook as a pre-development environment check.

## Package Importing

Check that the active notebook kernel can import the core packages: NumPy, PyTorch, OpenCV, rembg, and OpenAI.
If this cell fails, reinstall the environment from `requirements.txt`.

In [1]:
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF

import cv2
import rembg
import openai

from PIL import Image
import json

## DINOv3 Model Loading

Check that the DINOv3 source and Load local ViT-H+ weights.

In [2]:
from origami_texturing.paths import EXTERNAL_DIR
DINOV3_LOCAL_LOCATION = EXTERNAL_DIR / "dinov3"
DINOV3_GITHUB_LOCATION = "facebookresearch/dinov3"

if os.getenv("DINOV3_LOCATION") is not None:
    DINOV3_LOCATION = os.getenv("DINOV3_LOCATION")
elif DINOV3_LOCAL_LOCATION.exists():
    DINOV3_LOCATION = str(DINOV3_LOCAL_LOCATION)
else:
    DINOV3_LOCATION = DINOV3_GITHUB_LOCATION

print(f"DINOv3 location set to {DINOV3_LOCATION}")

DINOv3 location set to C:\Users\morph\GitHub\origami-texturing\external\dinov3


In [3]:
# examples of available DINOv3 models:
MODEL_DINOV3_VITS = "dinov3_vits16"
MODEL_DINOV3_VITSP = "dinov3_vits16plus"
MODEL_DINOV3_VITB = "dinov3_vitb16"
MODEL_DINOV3_VITL = "dinov3_vitl16"
MODEL_DINOV3_VITHP = "dinov3_vith16plus"
MODEL_DINOV3_VIT7B = "dinov3_vit7b16"

MODEL_NAME = MODEL_DINOV3_VITHP

from origami_texturing.paths import MODELS_DIR
DINOV3_WEIGHTS_PATH = str(MODELS_DIR / "dinov3" / "dinov3_vith16plus.pth")

model = torch.hub.load(
    repo_or_dir=DINOV3_LOCATION,
    model="dinov3_vith16plus",
    source="local",
    weights=DINOV3_WEIGHTS_PATH,
)
model.cuda()

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1280, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-31): 32 x SelfAttentionBlock(
      (norm1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=1280, out_features=3840, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1280, out_features=1280, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (mlp): SwiGLUFFN(
        (w1): Linear(in_features=1280, out_features=5120, bias=True)
        (w2): Linear(in_features=1280, out_features=5120, bias=True)
        (w3): Linear(in_features=5120, out_features=1280, bias=True)
      )
      (ls2): LayerScale()
    )
  )
  (norm): LayerNorm((